# 03 Decompression relay

Notebook for the decompression relay function.ast updated 31/05/2024)

Start with dependencies:

In [1]:
import numpy as np
import pyDEW
import os
from thermoengine import model
from copy import deepcopy
import pandas as pd
import linecache
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
import warnings

# Additional functions for pyDEW
import pyDEW_functions as pDF

# Structural functions for the relay
import relay_functions as rF

# Folder to export results
if not os.path.exists("./data"):
    os.makedirs("./data")

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

Create our pyDEW system (required for DATA0 and DATA1 files).

In [2]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)


Create the reactants that will fill our column.

In [3]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

mgtDict = {'reactant': 'MAGNETITE',
           'moles': 0.04}

startingReactant = [olvDict, opxDict, cpxDict, spnDict, mgtDict]
startingReactant = pDF.reactantToMass(startingReactant, 1000.0, dew_system)
startingReactantDF = pDF.reactantToDF(startingReactant, dew_system)


Create the column from the reactants.

In [4]:
# Create some example PT
steps = 3
temperatureLimits = [400+273.15, 500+273.15]  # in K
pressureLimits = [15000.0, 14000.0]  # in bar

temperatures = np.linspace(temperatureLimits[0], temperatureLimits[1], steps)
pressures = np.linspace(pressureLimits[0], pressureLimits[1], steps)

# Create the column dataframe
startingComposition = pd.concat([startingReactantDF]*len(pressures))
startingComposition['Pressure (bar)'] = pressures
startingComposition['Temperature (K)'] = temperatures
startingComposition

,FORSTERITE,FAYALITE,ENSTATITE_OR,FERROSILITE,DIOPSIDE(CPX),HEDENBERGITE(CPX),SPINEL,MAGNETITE,Pressure (bar),Temperature (K)
0,4.728685,0.525409,1.522797,0.1692,0.160294,0.01781,0.035621,0.035621,15000.0,673.15
0,4.728685,0.525409,1.522797,0.1692,0.160294,0.01781,0.035621,0.035621,14500.0,723.15
0,4.728685,0.525409,1.522797,0.1692,0.160294,0.01781,0.035621,0.035621,14000.0,773.15


Create the reactant fluid dictionary from interpolation from Guillaume's database.

In [5]:
fluidComposition = rF.createHybridFluid([0,0,1], 15000.0, 400+273.15)
fluidComposition

({'NA+': 0.0645091727471751,
  'CA+2': 0.000993810773977128,
  'MG+2': 0.000134994361838837,
  'AL+3': 0.00962141193444693,
  'H4SIO4(AQ)': 0.124337622326655,
  'CO3-2': 0.00100000002403092,
  'CL-': 0.010000000001451,
  'FE+2': 0.000623948173186246},
 -25.425807195652464)

## Decompression relay mark 2

In [6]:
def decompressionRelay(startingColumn, startingFluid, startingfO2, system, timesteps=10,
                       startingFRR=1.0, terminalFRR=0.2, molalityLimit=-20.0,
                       exportPath='data/export', outputToRead='working/output',
                       tabToRead='working/tab', **kwargs):
    """
    Decompression relay as a function of time. This models a column of rock getting flushed with a
    fluid periodically from the base of the rock column.

    INPUTS
    ------
    startingColumn : pandas.DataFrame
        pandas DataFrame with each column a different mineral composition (moles), with two
        additional columns 'Pressure (bar)' and 'Temperature (K)'
    startingFluid : dict
        dictionary of elemental molalities describing the composition of the flushing fluid.
        NB: this is separated from the rest of the function to allow for fluids from different
        pressures and temperatures to the first reaction
    startingfO2 : float
        fO2 of the fluid entering the system
    system : pyDEW.System object
        pyDEW.System for calculations
    timesteps : int
        number of times to flush our solid with the fluid. Default is 10
    startingFRR : float
        fluid-rock ratio (FRR) at the column base between first rock and fluid. Default is 1.0
    terminalFRR : float
        fluid-rock ratio (FRR) to terminate reactions. Default is 0.1
    molalityLimit : float
        log molality to consider a fluid exhausted in a particular element. Default is -20.0
    exportPath : string
        path to export xlsx file. Default is 'data/export'
    outputToRead : string
        path to pyDEW.Reaction output file. Default is 'working/output'
    tabToRead : string
        path to pyDEW.Reaction tab file. Default is 'working/tab'
    **kwargs for pyDEW.Fluid and pyDEW.Reaction functions

    OUTPUTS
    -------
    xlsx file, exported to exportPath
        Excel file with solid, element, and fluid speciation data

    """

    print('Relay function initiated')
    start = timer()
    t = 0
    allCombinedOutputs = []
    totalCalculations = 0
    while t < timesteps:
        if t == 0:
            workingColumn = startingColumn
        else:
            workingColumn = lastColumn

        # Track errors with these parameters
        calculations = 0
        passes = 0
        errors = 0
        outputs = []

        ##### DECOMPRESSION RELAY #####
        for n in range(len(workingColumn)):
            workingParameters = workingColumn.iloc[n]
            workingReactant = pDF.DFtoReactant(workingParameters, system)
            workingT = workingParameters['Temperature (K)']
            workingP = workingParameters['Pressure (bar)']

            ##### FLUID #####
            # At start, get parameters from the input, else use results from last reaction
            if n == 0:
                workingFRR = startingFRR
                if t == 0:
                    slabFluidMass = startingFRR * pDF.findReactantMass(workingReactant, system)
                workingFluidMass = slabFluidMass
                workingFluidMolalities = startingFluid
                workingfO2 = startingfO2
            else:
                reactionInput = deepcopy(reactionOutput)
                workingFRR = reactionInput['Properties']['final FRR']
                workingFluidMass = reactionInput['Properties']['final fluid mass']
                workingfO2 = reactionInput['Properties']['final fluid fO2']
                workingFluidMolalities = rF.formatFluid(reactionInput['Elements'])

                # Format from log molal to molal, check for very low molalities
                workingFluidMolalities.update(
                    (i, 10**j) for i, j in workingFluidMolalities.items())
                for i in workingFluidMolalities:
                    if workingFluidMolalities[i] < 10.0**(molalityLimit):
                        workingFluidMolalities[i] = 0.0

            # Suppress H2CO3(AQ) - shouldn't matter according to Guillaume
            nxmods = [['H2CO3(AQ)', 0, -1, 0]]
            workingFluid = pyDEW.Fluid(system, workingT, workingP,
                                       molalities=workingFluidMolalities, fO2=workingfO2,
                                       uebal='H+', uacion='CL-', nxmods=nxmods,
                                       aH2O_mode='molfraction')

            ##### REACTION #####
            try:
                # If we are below our FRR threshold, flag error
                if workingFRR < terminalFRR:
                    passes+=1
                    errorMessage = f'F/R threshold ({terminalFRR}) achieved at {n=},{t=}'
                    raise pyDEW.core.RunError(errorMessage)
                reactionOutput = rF.reactionByMass(workingReactant, workingFluid,
                                                   workingFluidMass, system,
                                                   workingP, workingT)
                calculations+=1
            except (KeyError, pyDEW.core.RunError):
                errors+=1
                reactionOutput = rF.failedReactionOutput(workingReactant, workingFluid,
                                                         workingFluidMass, system,
                                                         workingP, workingT)
            outputs.append(reactionOutput)
        combinedOutputs = rF.combineOutputs(outputs, system)
        allCombinedOutputs.append(combinedOutputs)
        totalCalculations+=calculations
        if timesteps - t > 1:
            lastColumn = deepcopy(combinedOutputs[1])
            lastColumn['Temperature (K)'] = combinedOutputs[0]['Temperature (K)'].values
            lastColumn['Pressure (bar)'] = combinedOutputs[0]['Pressure (bar)'].values
            print(f'{t=} is complete, {calculations=}, {errors=} of which {passes=}')
            t+=1
        else:
            rF.combineTimes(allCombinedOutputs, exportPath)
            print(f'{t=} is complete, {calculations=}, {errors=} of which {passes=}')
            print('-----RELAY COMPLETE-----')
            end = timer()
            print(f'Total iterations : {timesteps*len(startingColumn)}')
            print(f'Total successful DEW-EQ3/6 calculations : {totalCalculations}')
            print(f'Total runtime : {round(end-start, 1)} seconds')
            print(f'Total runtime : {round((end-start)/60, 1)} minutes')
            print(f'Data exported to: {exportPath}.xlsx')
            return

In [7]:
decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1], dew_system, timesteps=3)

Relay function initiated
t=0 is complete, calculations=3, errors=0 of which passes=0
t=1 is complete, calculations=3, errors=0 of which passes=0
t=2 is complete, calculations=3, errors=0 of which passes=0
-----RELAY COMPLETE-----
Total iterations : 9
Total successful DEW-EQ3/6 calculations : 9
Total runtime : 16.2 seconds
Total runtime : 0.3 minutes
Data exported to: data/export.xlsx
